<a href="https://colab.research.google.com/github/RDRamosU/cr-steam-comparativa-regional/blob/main/notebooks/02_exploracion_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto 4 — Costa Rica vs Latinoamérica en STEAM
## Notebook 02 — Exploración de datos

**Autor:** Ruben Dario Ramos Ulate
**Fecha:** Junio 2026  

---

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

print("Librerías importadas ✓")

Librerías importadas ✓


## 1. Carga de datasets UNESCO

In [ ]:
# Subir ambos archivos CSV de UNESCO
print("Selecciona los dos archivos CSV de UNESCO...")
uploaded = files.upload()
archivos = list(uploaded.keys())
print(f"Archivos cargados: {archivos}")

Selecciona los dos archivos CSV de UNESCO...


Saving UNESCO_UIS_GRAD_STEM.csv to UNESCO_UIS_GRAD_STEM.csv
Saving UNESCO_UIS_GRAD_ICT.csv to UNESCO_UIS_GRAD_ICT.csv
Archivos cargados: ['UNESCO_UIS_GRAD_STEM.csv', 'UNESCO_UIS_GRAD_ICT.csv']


In [ ]:
# Identificar archivos
archivo_stem = [f for f in archivos if 'STEM' in f.upper()][0]
archivo_ict  = [f for f in archivos if 'ICT' in f.upper()][0]

# Leer archivos
df_stem_raw = pd.read_csv(archivo_stem, skiprows=1)
df_stem_raw.columns = pd.read_csv(archivo_stem, nrows=0).columns

df_ict_raw = pd.read_csv(archivo_ict, skiprows=1)
df_ict_raw.columns = pd.read_csv(archivo_ict, nrows=0).columns

print(f"STEM: {df_stem_raw.shape} | ICT: {df_ict_raw.shape} ✓")

STEM: (6690, 39) | ICT: (6743, 39) ✓


## 2. Extracción de datos LAC — STEM

In [ ]:
# Países LAC del grupo comparativo
paises_lac = [
    'Costa Rica', 'Chile', 'Argentina', 'Colombia',
    'Mexico', 'Brazil', 'Peru', 'Uruguay',
    'Panama', 'Ecuador', 'Honduras', 'Guatemala'
]

def extraer_lac(df_raw, indicador):
    df = df_raw.rename(columns={
        'REF_AREA_LABEL': 'pais',
        'TIME_PERIOD': 'año',
        'OBS_VALUE': 'valor',
        'SEX_LABEL': 'sexo',
        'UNIT_MEASURE_LABEL': 'unidad'
    })
    # Solo porcentajes (no índices) y países LAC
    return df[
        (df['pais'].isin(paises_lac)) &
        (df['unidad'] == 'Percentage of graduates')
    ][['pais', 'año', 'sexo', 'valor']].dropna()

df_stem = extraer_lac(df_stem_raw, 'STEM')
df_ict  = extraer_lac(df_ict_raw,  'ICT')

print(f"STEM filtrado: {df_stem.shape}")
print(f"ICT filtrado:  {df_ict.shape}")
print(f"\nSexos disponibles: {df_stem['sexo'].unique()}")
print(f"Años disponibles: {sorted(df_stem['año'].unique())}")

STEM filtrado: (419, 4)
ICT filtrado:  (419, 4)

Sexos disponibles: ['Total' 'Female' 'Male']
Años disponibles: [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


## 3. Dato más reciente por país — STEM total

In [ ]:
# Dato más reciente por país — total ambos sexos
df_stem_total = df_stem[df_stem['sexo'] == 'Total']
df_stem_reciente = (df_stem_total
    .sort_values('año', ascending=False)
    .groupby('pais')
    .first()
    .reset_index()
    .sort_values('valor', ascending=False)
)

print("=== % GRADUADOS STEM DEL TOTAL TERCIARIO — DATO MÁS RECIENTE ===\n")
print(df_stem_reciente[['pais', 'año', 'valor']].to_string(index=False))

print(f"\nPromedio LAC: {df_stem_reciente['valor'].mean():.2f}%")
print(f"Costa Rica:   {df_stem_reciente[df_stem_reciente['pais']=='Costa Rica']['valor'].values[0]:.2f}%")

=== % GRADUADOS STEM DEL TOTAL TERCIARIO — DATO MÁS RECIENTE ===

      pais  año     valor
      Peru 2017 29.642820
  Colombia 2021 23.911249
    Mexico 2022 23.748711
     Chile 2022 21.382799
   Ecuador 2022 19.666300
    Brazil 2022 16.266979
Costa Rica 2023 15.775070
  Honduras 2019 15.732690
 Argentina 2022 14.809300
   Uruguay 2022 14.540020
    Panama 2022 13.020700
 Guatemala 2015  9.765440

Promedio LAC: 18.19%
Costa Rica:   15.78%


## 4. Brecha de género en graduados STEM por país

In [ ]:
# Brecha de género — año más reciente por país
df_stem_genero = df_stem[df_stem['sexo'].isin(['Female', 'Male'])]
df_stem_genero_reciente = (df_stem_genero
    .sort_values('año', ascending=False)
    .groupby(['pais', 'sexo'])
    .first()
    .reset_index()
)

# Pivot para comparar
df_genero_pivot = df_stem_genero_reciente.pivot(
    index='pais', columns='sexo', values='valor'
).reset_index()
df_genero_pivot.columns.name = None
df_genero_pivot['brecha_genero'] = (
    df_genero_pivot['Male'] - df_genero_pivot['Female']
).round(2)
df_genero_pivot = df_genero_pivot.sort_values('brecha_genero')

print("=== BRECHA DE GÉNERO EN STEM POR PAÍS (Male% - Female%) ===\n")
print(df_genero_pivot[['pais', 'Female', 'Male', 'brecha_genero']].to_string(index=False))

cr_brecha = df_genero_pivot[df_genero_pivot['pais']=='Costa Rica']['brecha_genero'].values[0]
print(f"\nBrecha CR: {cr_brecha:.2f} puntos porcentuales")
print(f"Promedio brecha LAC: {df_genero_pivot['brecha_genero'].mean():.2f} pp")

=== BRECHA DE GÉNERO EN STEM POR PAÍS (Male% - Female%) ===

      pais   Female      Male  brecha_genero
 Guatemala  5.43075 16.953430          11.52
   Uruguay 10.39879 22.504850          12.11
      Peru 24.44179 36.820339          12.38
 Argentina 10.58690 23.704281          13.12
    Panama  8.17287 22.593090          14.42
Costa Rica  9.38601 25.239950          15.85
  Honduras  9.40582 26.133801          16.73
    Brazil  8.56560 28.077480          19.51
  Colombia 15.08117 35.434540          20.35
   Ecuador 10.41161 31.427111          21.02
    Mexico 14.25955 35.865749          21.61
     Chile  7.80348 39.653912          31.85

Brecha CR: 15.85 puntos porcentuales
Promedio brecha LAC: 17.54 pp


## 5. Tendencia histórica de CR vs LAC — STEM

In [ ]:
# Tendencia 2015-2023 para países con datos completos
años_analisis = list(range(2015, 2024))

df_tendencia = df_stem_total[
    df_stem_total['año'].isin(años_analisis)
][['pais', 'año', 'valor']]

# Países con más datos disponibles
cobertura = df_tendencia.groupby('pais')['año'].count()
paises_completos = cobertura[cobertura >= 5].index.tolist()

print("=== TENDENCIA STEM 2015-2023 — PAÍSES CON ≥5 AÑOS DE DATOS ===\n")
df_pivot_tendencia = (df_tendencia[df_tendencia['pais'].isin(paises_completos)]
    .pivot(index='año', columns='pais', values='valor')
    .round(2))
print(df_pivot_tendencia.to_string())

=== TENDENCIA STEM 2015-2023 — PAÍSES CON ≥5 AÑOS DE DATOS ===

pais  Brazil  Chile  Colombia  Costa Rica  Ecuador  Honduras  Mexico  Panama  Uruguay
año                                                                                  
2015   15.35  20.10     22.70       12.88    16.69     14.72   27.90   17.24      NaN
2016   16.86  19.90     23.64       13.09    15.84     15.46   25.48   15.43      NaN
2017   17.73  20.47     23.66       14.40      NaN     15.02   25.24     NaN    17.45
2018   18.37  20.95     23.13       15.46     9.38     15.20     NaN     NaN    18.98
2019   18.53  20.57     24.63       15.11    16.24     15.73   25.83   16.46    17.22
2020   17.50  21.41     23.52       16.23    18.30       NaN   25.82   13.74    15.24
2021     NaN  20.51     23.91       15.72      NaN       NaN   24.34   15.23    18.57
2022   16.27  21.38       NaN       15.78    19.67       NaN   23.75   13.02    14.54
2023     NaN    NaN       NaN       15.78      NaN       NaN     NaN     NaN

## 6. Datos ICT — porcentaje graduados en TIC

In [ ]:
# ICT más reciente por país
df_ict_total = df_ict[df_ict['sexo'] == 'Total']
df_ict_reciente = (df_ict_total
    .sort_values('año', ascending=False)
    .groupby('pais')
    .first()
    .reset_index()
    .sort_values('valor', ascending=False)
)

print("=== % GRADUADOS ICT DEL TOTAL TERCIARIO — DATO MÁS RECIENTE ===\n")
print(df_ict_reciente[['pais', 'año', 'valor']].to_string(index=False))

cr_ict = df_ict_reciente[df_ict_reciente['pais']=='Costa Rica']['valor'].values[0]
print(f"\nCosta Rica ICT: {cr_ict:.2f}%")
print(f"Promedio LAC ICT: {df_ict_reciente['valor'].mean():.2f}%")

=== % GRADUADOS ICT DEL TOTAL TERCIARIO — DATO MÁS RECIENTE ===

      pais  año   valor
      Peru 2017 5.79122
Costa Rica 2023 5.48845
    Brazil 2022 4.62050
    Mexico 2022 4.23768
   Uruguay 2022 4.07174
  Colombia 2021 3.64831
    Panama 2022 3.51453
  Honduras 2019 3.43449
     Chile 2022 3.18680
   Ecuador 2022 2.43059
 Argentina 2022 1.65571
 Guatemala 2015 1.43705

Costa Rica ICT: 5.49%
Promedio LAC ICT: 3.63%


## 7. Nota metodológica importante

In [ ]:
print("=" * 60)
print("NOTA METODOLÓGICA")
print("=" * 60)
print("""
El indicador UNESCO 'Percentage of graduates from STEM
programmes in tertiary education' usa la clasificación
ISCED-F 2013, que incluye:
  - Ciencias naturales, matemáticas y estadística
  - Tecnologías de información y comunicación
  - Ingeniería, manufactura y construcción

Este indicador es más estrecho que la clasificación STEAM
usada por OPES-CONARE (que incluye Arte/Diseño).

Por eso UNESCO reporta CR con ~15.8% mientras OPES-CONARE
reporta ~34.1%. Ambos son correctos — miden clasificaciones
diferentes. En el análisis comparativo usamos UNESCO para
garantizar comparabilidad internacional.
""")

# Exportar datasets
df_stem_reciente.to_csv("stem_lac_reciente.csv", index=False)
df_genero_pivot.to_csv("stem_genero_lac.csv", index=False)
df_ict_reciente.to_csv("ict_lac_reciente.csv", index=False)
df_pivot_tendencia.to_csv("stem_tendencia_2015_2023.csv")

print("Datasets exportados ✓")

NOTA METODOLÓGICA

El indicador UNESCO 'Percentage of graduates from STEM 
programmes in tertiary education' usa la clasificación 
ISCED-F 2013, que incluye:
  - Ciencias naturales, matemáticas y estadística
  - Tecnologías de información y comunicación
  - Ingeniería, manufactura y construcción

Este indicador es más estrecho que la clasificación STEAM 
usada por OPES-CONARE (que incluye Arte/Diseño).

Por eso UNESCO reporta CR con ~15.8% mientras OPES-CONARE 
reporta ~34.1%. Ambos son correctos — miden clasificaciones 
diferentes. En el análisis comparativo usamos UNESCO para 
garantizar comparabilidad internacional.

Datasets exportados ✓
